In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os 

os.makedirs('../outputs/images', exist_ok=True)

df = pd.read_csv('../data/preprocessed/final.csv')
df.drop(['Unnamed: 0'], axis=1, inplace=True)

X = df.drop(columns=['loan_status'])
y = df['loan_status']


from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
import warnings
warnings.filterwarnings('ignore')
import time

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder 
from sklearn.preprocessing import StandardScaler , MinMaxScaler
from sklearn.metrics import mean_squared_error, r2_score , mean_absolute_error 
from sklearn.model_selection import StratifiedKFold, cross_validate

# Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import (
    BernoulliNB,
    CategoricalNB,
    ComplementNB,
    GaussianNB,
    MultinomialNB
)
from sklearn.tree import DecisionTreeClassifier , ExtraTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    BaggingClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    VotingClassifier,
    StackingClassifier
)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [8]:
# 5-Fold Stratified Cross-Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring_metrics = ['accuracy', 'precision']

clf_lr = LogisticRegression(max_iter=1000, random_state=42)
clf_rf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
clf_lgbm = LGBMClassifier(n_estimators=50, random_state=42, verbose=-1, n_jobs=-1)


models = {  "Voting Classifier": VotingClassifier(
        estimators=[('lr', clf_lr), ('rf', clf_rf), ('lgb', clf_lgbm)],
        voting='soft'
    )
}
results = []
all_pipelines = {}

for name, model in models.items():
    print(f"🔄 Running 5-Fold CV for {name}...")
    start_time = time.time()
    
    pipe = Pipeline([
        ('model', model)
    ])
    
    try:
        cv_scores = cross_validate(
            pipe,
            X,
            y,
            cv=skf,
            scoring=scoring_metrics,
            n_jobs=-1,
            error_score='raise'
        )
        elapsed_time = time.time() - start_time
        all_pipelines[name] = pipe

        results.append({
            'Algorithm': name,
            'Mean_Accuracy': np.round(np.mean(cv_scores['test_accuracy']), 4),
            'Std_Accuracy': np.round(np.std(cv_scores['test_accuracy']), 4),
            'Mean_Precision': np.round(np.mean(cv_scores['test_precision']), 4),
            'Std_Precision': np.round(np.std(cv_scores['test_precision']), 4),
            'CV_Time_s': round(elapsed_time, 2)
        })
        print(f"✓ Completed {name} in {round(elapsed_time, 2)}s")
    except Exception as e:
        print(f"✗ Failed {name}: {e}")

# Print results outside the loop
performance_df = pd.DataFrame(results)
print("\n" + "=" * 50)
print(performance_df.to_string(index=False))
print("=" * 50)

🔄 Running 5-Fold CV for Voting Classifier...
✓ Completed Voting Classifier in 13.49s

        Algorithm  Mean_Accuracy  Std_Accuracy  Mean_Precision  Std_Precision  CV_Time_s
Voting Classifier         0.9843        0.0045           0.995         0.0051      13.49


In [11]:
import optuna
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from lightgbm import LGBMClassifier

In [12]:
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

In [13]:
# 2. Optuna Objective Function

def objective(trial):
    # --- 1. Logistic Regression Hyperparameters ---
    lr_c = trial.suggest_float('lr_C', 1e-4, 10.0, log=True)
    clf_lr = LogisticRegression(
        C=lr_c,
        max_iter=1000,
        random_state=42
    )

    # --- 2. Random Forest Hyperparameters ---
    rf_n_estimators = trial.suggest_int('rf_n_estimators', 30, 150, step=20)
    rf_max_depth = trial.suggest_int('rf_max_depth', 4, 15)
    rf_min_samples_split = trial.suggest_int('rf_min_samples_split', 2, 10)
    clf_rf = RandomForestClassifier(
        n_estimators=rf_n_estimators,
        max_depth=rf_max_depth,
        min_samples_split=rf_min_samples_split,
        random_state=42,
        n_jobs=-1
    )

    # --- 3. LightGBM Hyperparameters ---
    lgbm_n_estimators = trial.suggest_int('lgbm_n_estimators', 30, 150, step=20)
    lgbm_learning_rate = trial.suggest_float('lgbm_learning_rate', 0.01, 0.2, log=True)
    lgbm_num_leaves = trial.suggest_int('lgbm_num_leaves', 15, 63)
    clf_lgbm = LGBMClassifier(
        n_estimators=lgbm_n_estimators,
        learning_rate=lgbm_learning_rate,
        num_leaves=lgbm_num_leaves,
        random_state=42,
        verbose=-1,
        n_jobs=-1
    )

    # --- 4. Voting Classifier Weights ---
    w_lr = trial.suggest_int('weight_lr', 1, 5)
    w_rf = trial.suggest_int('weight_rf', 1, 5)
    w_lgbm = trial.suggest_int('weight_lgbm', 1, 5)

    voting_clf = VotingClassifier(
        estimators=[
            ('lr', clf_lr),
            ('rf', clf_rf),
            ('lgbm', clf_lgbm)
        ],
        voting='soft',   # 'soft' uses predicted probabilities
        weights=[w_lr, w_rf, w_lgbm],
        n_jobs=-1
    )

    # 5-Fold Stratified Cross-Validation
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(voting_clf, X_scaled, y, cv=cv, scoring='accuracy', n_jobs=-1)

    return scores.mean()


optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=35, timeout=600)

print("=" * 50)
print(f"Best Mean CV Accuracy: {study.best_value:.4f}")
print("Best Hyperparameters:")
for param, val in study.best_params.items():
    print(f"  {param}: {val}")
print("=" * 50)


Best Mean CV Accuracy: 0.9871
Best Hyperparameters:
  lr_C: 9.507375237278625
  rf_n_estimators: 90
  rf_max_depth: 15
  rf_min_samples_split: 8
  lgbm_n_estimators: 150
  lgbm_learning_rate: 0.17773120245786644
  lgbm_num_leaves: 63
  weight_lr: 2
  weight_rf: 5
  weight_lgbm: 5


In [17]:
best = study.best_params

final_lr = LogisticRegression(
    C=best['lr_C'], 
    max_iter=1000, 
    random_state=42
)
final_rf = RandomForestClassifier(
    n_estimators=best['rf_n_estimators'],
    max_depth=best['rf_max_depth'],
    min_samples_split=best['rf_min_samples_split'],
    random_state=42,
    n_jobs=-1
)
final_lgbm = LGBMClassifier(
    n_estimators=best['lgbm_n_estimators'],
    learning_rate=best['lgbm_learning_rate'],
    num_leaves=best['lgbm_num_leaves'],
    random_state=42,
    verbose=-1,
    n_jobs=-1
)

final_voting_clf = VotingClassifier(
    estimators=[('lr', final_lr), ('rf', final_rf), ('lgbm', final_lgbm)],
    voting='soft',
    weights=[best['weight_lr'], best['weight_rf'], best['weight_lgbm']],
    n_jobs=-1
)

final_voting_clf.fit(X_scaled, y)
print("\nFinal Voting Classifier trained successfully.")
print (f"Best Hyperparameters: {best}")
print("\nAccuracy on Training Set: {:.4f}".format(final_voting_clf.score(X_scaled, y)))
print("\nPrecision on Training Set: {:.4f}".format(cross_val_score(final_voting_clf, X_scaled, y, cv=5, scoring='precision').mean()))


Final Voting Classifier trained successfully.
Best Hyperparameters: {'lr_C': 9.507375237278625, 'rf_n_estimators': 90, 'rf_max_depth': 15, 'rf_min_samples_split': 8, 'lgbm_n_estimators': 150, 'lgbm_learning_rate': 0.17773120245786644, 'lgbm_num_leaves': 63, 'weight_lr': 2, 'weight_rf': 5, 'weight_lgbm': 5}

Accuracy on Training Set: 1.0000

Precision on Training Set: 0.9857


In [18]:
from sklearn.model_selection import cross_validate

# Real 5-Fold Validation Score dekhein
cv_results = cross_validate(
    final_voting_clf, 
    X_scaled, 
    y, 
    cv=5, 
    scoring=['accuracy', 'precision', 'recall', 'f1']
)

print(f"CV Mean Accuracy : {cv_results['test_accuracy'].mean():.4f}")
print(f"CV Mean Precision: {cv_results['test_precision'].mean():.4f}")
print(f"CV Mean Recall   : {cv_results['test_recall'].mean():.4f}")
print(f"CV Mean F1-Score : {cv_results['test_f1'].mean():.4f}")

CV Mean Accuracy : 0.9866
CV Mean Precision: 0.9857
CV Mean Recall   : 0.9789
CV Mean F1-Score : 0.9823


In [19]:
import joblib
import os

model_path = '../models/loan_approval_voting_classifier.joblib'
os.makedirs('../models', exist_ok=True)

joblib.dump(
    {
        'model': final_voting_clf,
        'scaler': scaler,
        'feature_names': list(X.columns)
    },
    model_path
)
print(f'Model saved to: {model_path}')

Model saved to: ../models/loan_approval_voting_classifier.joblib
